# CMP7005 Programming for Data Analysis Reassessment

## From Data to Application Development

This notebook documents the complete workflow for the Indian air quality dataset, from data handling and exploratory analysis to model building and application development. All numerical outputs should be generated by running the code cells against the supplied dataset.


## Project Objectives

The objectives are to load and inspect the supplied air quality data, clean and preprocess it responsibly, investigate meaningful air pollution patterns, build and evaluate regression models for AQI prediction, and support the analysis through a Streamlit application.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.data_loader import load_combined_dataset, save_combined_dataset, inspect_dataset
from src.preprocessing import clean_air_quality_data, POLLUTANT_COLUMNS
from src.eda_functions import city_aqi_summary, yearly_aqi_summary, monthly_aqi_summary, pollutant_correlations, missing_value_summary
from src.model import train_and_compare_models, save_model
from src.utils import RAW_DATA_DIR, COMBINED_DATA_PATH, MODEL_PATH, RESULTS_PATH


## Task 1: Data Handling

The dataset is supplied as separate city CSV files. These files are loaded, validated against the expected schema, combined into one dataset, and saved as `data/processed/air_quality.csv` for reproducible use by the notebook and application.


In [ ]:
data = save_combined_dataset(RAW_DATA_DIR, COMBINED_DATA_PATH)
inspection = inspect_dataset(data)
inspection['shape'], inspection['date_min'], inspection['date_max'], len(inspection['cities'])


In [ ]:
display(data.head())
display(data.tail())
display(pd.DataFrame({'dtype': data.dtypes.astype(str), 'missing': data.isna().sum(), 'unique': data.nunique(dropna=False)}))
print('Duplicate records:', data.duplicated().sum())
print('Cities:', inspection['cities'])


## Task 2: Exploratory Data Analysis

EDA receives the greatest assessment weighting. The following sections examine fundamental data understanding, preprocessing, and univariate, bivariate, and multivariate relationships. Interpretations should be written after running each output so that claims are supported by actual results.


### 2.1 Fundamental Data Understanding


In [ ]:
clean_data = clean_air_quality_data(data)
display(missing_value_summary(clean_data))
display(clean_data['City'].value_counts().to_frame('Records'))
display(clean_data['AQI_Bucket'].value_counts(dropna=False).to_frame('Records'))
display(clean_data[['AQI'] + POLLUTANT_COLUMNS].describe().T)


### 2.2 Data Preprocessing

The preprocessing keeps the original raw data separate. Duplicate rows are removed, dates are parsed, pollutant columns are converted to numeric values, negative pollutant values are treated as invalid, and year, month, month name, and season features are created for time-based analysis. Missing pollutant values are not blindly replaced during EDA; they are inspected and handled according to the analysis or model requirement.


In [ ]:
display(clean_data[['Date', 'Year', 'Month', 'Month_Name', 'Season']].head())
print('Rows before cleaning:', len(data))
print('Rows after duplicate removal and feature creation:', len(clean_data))


### 2.3 Univariate Analysis


In [ ]:
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(clean_data['AQI'].dropna(), bins=40, kde=True, ax=axes[0])
axes[0].set_title('Distribution of AQI')
sns.countplot(data=clean_data, y='AQI_Bucket', order=clean_data['AQI_Bucket'].value_counts().index, ax=axes[1])
axes[1].set_title('AQI Category Distribution')
plt.tight_layout()


### 2.4 Bivariate Analysis


In [ ]:
city_summary = city_aqi_summary(clean_data)
display(city_summary.head(10))
display(city_summary.tail(10))

plt.figure(figsize=(10, 6))
sns.barplot(data=city_summary.head(15), x='Mean_AQI', y='City')
plt.title('Highest Average AQI by City')
plt.xlabel('Mean AQI')
plt.ylabel('City')
plt.tight_layout()


In [ ]:
display(yearly_aqi_summary(clean_data))
display(monthly_aqi_summary(clean_data))

plt.figure(figsize=(9, 5))
sns.lineplot(data=yearly_aqi_summary(clean_data), x='Year', y='Mean_AQI', marker='o')
plt.title('Average AQI by Year')
plt.ylabel('Mean AQI')
plt.tight_layout()


In [ ]:
pollutant = 'PM2.5'
plt.figure(figsize=(8, 5))
sns.scatterplot(data=clean_data, x=pollutant, y='AQI', alpha=0.4)
plt.title(f'{pollutant} Compared with AQI')
plt.tight_layout()


### 2.5 Multivariate Analysis


In [ ]:
corr = pollutant_correlations(clean_data)
display(corr['AQI'].sort_values(ascending=False))
plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap for Pollutants and AQI')
plt.tight_layout()


## Task 3: Model Building

The modelling objective is to predict numerical AQI using pollutant measurements, city, year, and month. `AQI_Bucket` is excluded because it is derived from AQI and would cause target leakage.


In [ ]:
best_model, model_results, split_data = train_and_compare_models(clean_data)
display(model_results)
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
model_results.to_csv(RESULTS_PATH, index=False)
save_model(best_model, MODEL_PATH)
MODEL_PATH


## Task 4: Application Development

The project includes a Streamlit application in `app/app.py` with three sections: Data Overview, Exploratory Data Analysis, and Modelling and Prediction. The app uses the reusable modules in `src/` and the saved model in `models/trained_model.pkl`.

Run it with:

```bash
streamlit run app/app.py
```


## Task 5: Version Control

A GitHub repository has been created for this project. Include evidence of regular development commits, the repository structure, and the pushed project files.

Repository URL: `https://github.com/GursevakSingh-ui/CMP_7005_Prac1.git`

Screenshots to add in `assets/screenshots/`:

- GitHub repository file structure
- GitHub commit history
- Streamlit Data Overview page
- Streamlit EDA page
- Streamlit Prediction page


## Task 6: Self Reflection

Complete this section in the student's own words before submission. The reflection should be honest and based on the actual development process. Suggested points to cover are listed below.

### Reflection Draft Placeholder

- Main challenges encountered during the project
- Technical difficulties with Python, data processing, modelling, Streamlit, or GitHub
- Data quality issues such as missing values and separate city files
- EDA decisions and which charts were most useful
- Model-building challenges, including avoiding target leakage from `AQI_Bucket`
- Application development challenges
- Version-control learning and GitHub workflow
- Skills developed during the project
- How the project improved during development
- What would be improved next time

Previous feedback placeholder: add genuine previous assessment feedback here if supplied. If no feedback was supplied, state that no previous assessment feedback was available.


## AI Use Declaration

AI support was used to help structure, code, and review the project. The student must review, understand, test, and adapt the work before submission. All final reflection details and assessment-specific claims should be completed honestly by the student.


## Final Compliance Audit

| Assessment Requirement | Implemented | Evidence or File | Remaining Action |
|---|---|---|---|
| Data imported and inspected | Yes | `src/data_loader.py`, notebook Task 1 | Add written interpretation after running notebook |
| Missing values and duplicates assessed | Yes | `src/eda_functions.py`, notebook Task 2 | Explain chosen handling in final narrative |
| Feature engineering applied | Yes | `src/preprocessing.py` | Justify date features in notebook |
| Univariate EDA | Yes | Notebook Task 2.3, app EDA page | Add interpretation text after outputs |
| Bivariate EDA | Yes | Notebook Task 2.4, app EDA page | Add interpretation text after outputs |
| Multivariate EDA | Yes | Notebook Task 2.5 | Add interpretation text after outputs |
| Model comparison completed | Yes | `models/model_comparison.csv`, notebook Task 3 | Discuss limitations and overfitting risk |
| GUI developed | Yes | `app/app.py`, `app/pages/` | Capture screenshots after testing |
| README and requirements | Yes | `README.md`, `requirements.txt` | Update with GitHub link when available |
| GitHub evidence | Partial | `.gitignore`, README guidance | Create repository, commit history, screenshots |
| Reflection | Partial | Notebook placeholder | Student must complete genuine reflection |
| AI use acknowledged | Drafted | Notebook AI declaration | Adjust to university wording if required |
